# Dealer descriptives: USD Treasury vs EUR euro-area sovereign repo

Universe (set in the setup cell): single-security **specific** government collateral
(`gnlcoll = 'SPEC'`, `security_type = 'GOVS'`, `collateral_basket_id IS NULL`),
**US Treasuries funded in USD** vs the large **euro-area sovereigns (DE, FR, IT, ES)
funded in EUR** (no cross-currency/collateral mix). Amounts use `nominal_euro`, so both
markets are expressed in EUR.

Volumes are **average daily outstanding** (daily sum per dealer divided by that market's
number of business days), in EUR bn. Tables and concentration drop dealers below a
`MIN_OUTSTANDING` noise floor; the scatter plots restrict to the largest `TOP_N` dealers
(and winsorise where noted) to keep the axes readable.

Direction follows the cash leg: `borrower_id` is the **cash borrower = collateral lender**
(finances a long bond inventory), `lender_id` is the **cash lender = collateral borrower**
(reverse repo). Net is `coll_lender - coll_borrower`, so a positive net flags a net supplier
of collateral / net financer of a long position.

Run the setup cell first; the four sections below are independent of one another.

In [ ]:
import pyodbc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

cnxn = pyodbc.connect('DSN=Hermes_DSN', autocommit=True)

MIN_OUTSTANDING = 1e7  # EUR 10m average daily outstanding floor (tables, concentration)
TOP_N = 100            # largest-N dealers shown in the scatter plots

BASE_QUERY = """
WITH base AS (
    SELECT business_date, borrower_id, lender_id,
           LEFT(security_isin, 2) AS coll_country, nominal_euro
    FROM xlab_ecb_prj_sftds_cb_common.hermesf_state
    WHERE gnlcoll = 'SPEC'
      AND collateral_basket_id IS NULL
      AND security_type = 'GOVS'
      AND ((LEFT(security_isin, 2) = 'US' AND nominal_ccy = 'USD')
           OR (LEFT(security_isin, 2) IN ('DE', 'FR', 'IT', 'ES') AND nominal_ccy = 'EUR'))
),
legs AS (
    SELECT coll_country, borrower_id AS dealer, 'coll_lender' AS side, nominal_euro FROM base
    UNION ALL
    SELECT coll_country, lender_id AS dealer, 'coll_borrower' AS side, nominal_euro FROM base
),
days AS (
    SELECT coll_country, COUNT(DISTINCT business_date) AS n_days FROM base GROUP BY coll_country
)
SELECT l.coll_country, l.dealer, l.side,
       SUM(l.nominal_euro) / MAX(d.n_days) AS volume
FROM legs l
JOIN days d ON l.coll_country = d.coll_country
GROUP BY l.coll_country, l.dealer, l.side
"""

## 1. Per-dealer collateral mix and US-collateral share

For each dealer: average daily outstanding in US Treasuries (`us`, USD) and in euro-area
sovereigns (`ea`, EUR), the US-collateral share `us / (us + ea)`, and the net position.
Sorted by total size; the share is the cross-sectional measure of US-collateral exposure.
Volumes in EUR bn.

In [ ]:
df = pd.read_sql_query(BASE_QUERY, cnxn)

df['market'] = np.where(df['coll_country'] == 'US', 'us', 'ea')
by_market = df.pivot_table(index='dealer', columns='market', values='volume',
                           aggfunc='sum', fill_value=0).reindex(columns=['us', 'ea'], fill_value=0)
by_side = df.pivot_table(index='dealer', columns='side', values='volume',
                         aggfunc='sum', fill_value=0).reindex(columns=['coll_lender', 'coll_borrower'], fill_value=0)

dealers = by_market.join(by_side)
dealers['total'] = dealers['us'] + dealers['ea']
dealers['us_share'] = dealers['us'] / dealers['total']
dealers['net'] = dealers['coll_lender'] - dealers['coll_borrower']
dealers = dealers[dealers['total'] >= MIN_OUTSTANDING].sort_values('total', ascending=False)

table = dealers[['us', 'ea', 'us_share', 'net']].copy()
table[['us', 'ea', 'net']] /= 1e9
table.head(20)

## 2. Dealer overlap: US Treasuries vs euro-area sovereigns

Do the dealers that are large in US Treasury repo also intermediate euro-area sovereign
repo? Average daily outstanding per dealer in each market, restricted to the largest
`TOP_N` dealers so the scatter stays readable, with the rank (Spearman) and log-level
(Pearson) correlations and the count active in both.

In [ ]:
df = pd.read_sql_query(BASE_QUERY, cnxn)

df['market'] = np.where(df['coll_country'] == 'US', 'US', 'EA')
g = df.groupby(['dealer', 'market'])['volume'].sum().unstack(fill_value=0)
g = g.reindex(columns=['US', 'EA'], fill_value=0)
g['total'] = g['US'] + g['EA']
g = g.sort_values('total', ascending=False).head(TOP_N)

both = g[(g['US'] > 0) & (g['EA'] > 0)]
spearman = g['US'].corr(g['EA'], method='spearman')
pearson_log = np.corrcoef(np.log10(both['US']), np.log10(both['EA']))[0, 1]

print(f"top {TOP_N} dealers, active in both: {len(both)}, "
      f"Spearman: {spearman:.2f}, Pearson (log, both): {pearson_log:.2f}")

plt.figure(figsize=(6, 6))
plt.scatter(both['US'] / 1e9, both['EA'] / 1e9, s=12, alpha=0.5)
plt.xscale('log')
plt.yscale('log')
plt.xlabel('US Treasuries, avg daily outstanding (EUR bn)')
plt.ylabel('EA sovereigns, avg daily outstanding (EUR bn)')
plt.title(f'Dealer overlap, top {TOP_N}')
plt.show()

## 3. Concentration

Top-5 / top-10 dealer shares and the HHI (0-10,000) for the US and the euro-area markets
(active dealers only), with the cumulative-share curve by dealer rank.

In [ ]:
df = pd.read_sql_query(BASE_QUERY, cnxn)

df['market'] = np.where(df['coll_country'] == 'US', 'US', 'EA')
g = df.groupby(['market', 'dealer'])['volume'].sum().reset_index()
g = g[g['volume'] >= MIN_OUTSTANDING]

rows = []
for market, sub in g.groupby('market'):
    s = (sub['volume'] / sub['volume'].sum()).sort_values(ascending=False)
    rows.append({'market': market, 'n_dealers': len(s),
                 'top5_share': s.head(5).sum(), 'top10_share': s.head(10).sum(),
                 'hhi': (s ** 2).sum() * 1e4})
concentration = pd.DataFrame(rows).set_index('market')

plt.figure(figsize=(6, 4))
for market, sub in g.groupby('market'):
    s = (sub['volume'] / sub['volume'].sum()).sort_values(ascending=False).cumsum().values
    plt.plot(range(1, len(s) + 1), s, label=market)
plt.xlabel('dealer rank')
plt.ylabel('cumulative share')
plt.legend()
plt.title('Concentration of sovereign repo by dealer')
plt.show()

concentration

## 4. Net versus gross

For US Treasuries, net against gross for the largest `TOP_N` dealers (net winsorised at the
1st/99th percentile to tame outliers) and the distribution of directionality (`net / gross`,
in [-1, 1]): +1 a pure collateral lender financing a long inventory, -1 a pure collateral
borrower (reverse repo), around 0 a two-way intermediary.

In [ ]:
df = pd.read_sql_query(BASE_QUERY, cnxn)

us = df[df['coll_country'] == 'US']
piv = us.pivot_table(index='dealer', columns='side', values='volume',
                     aggfunc='sum', fill_value=0)
piv = piv.reindex(columns=['coll_lender', 'coll_borrower'], fill_value=0)
piv['gross'] = piv['coll_lender'] + piv['coll_borrower']
piv['net'] = piv['coll_lender'] - piv['coll_borrower']
piv['directionality'] = piv['net'] / piv['gross']
piv = piv.sort_values('gross', ascending=False).head(TOP_N)

lo, hi = piv['net'].quantile([0.01, 0.99])
net_w = piv['net'].clip(lo, hi) / 1e9

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(piv['gross'] / 1e9, net_w, s=12, alpha=0.5)
ax[0].axhline(0, color='k', lw=0.6)
ax[0].set_xscale('log')
ax[0].set_xlabel('gross (EUR bn)')
ax[0].set_ylabel('net, winsorised 1/99% (EUR bn)')
ax[0].set_title(f'US Treasury repo: net vs gross, top {TOP_N}')

ax[1].hist(piv['directionality'], bins=25)
ax[1].set_xlabel('directionality = net / gross')
ax[1].set_ylabel('dealers')
ax[1].set_title('Directionality of US Treasury repo')
plt.tight_layout()
plt.show()